In [1]:
import ir_datasets
dataset = ir_datasets.load("wikir/en1k/training")
doc_generator = (doc.text for doc in dataset.docs_iter())
print("docs generator created!")

docs generator created!


In [2]:
from nltk.stem import PorterStemmer

porter_stemmer = PorterStemmer()
stemmed_words = [porter_stemmer.stem(word) for word in doc_generator]
print("stemmed words ready!")

stemmed words ready!


### BM25

In [3]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [doc.split(" ") for doc in stemmed_words]
bm25 = BM25Okapi(tokenized_corpus)

In [4]:
queries = [query.text for query in dataset.queries_iter()]

scores_of_all_queries = []

for query in queries:
    scores_of_all_queries.append(bm25.get_scores(query.split(" ")))

In [5]:
from collections import defaultdict
from helper import Scoredoc

doc_dict = defaultdict(str)

for i, doc in enumerate(dataset.docs_iter()):
    doc_dict[i] = doc.doc_id

doc_dict = dict(doc_dict)

qrels_dict = defaultdict(list)

for qrel in dataset.qrels_iter():
    qrels_dict[qrel.query_id].append(qrel.doc_id)

qrels_dict = dict(qrels_dict)

score_doc_dict = defaultdict(list)

for scoreddoc in dataset.scoreddocs_iter():
    doc_id = scoreddoc.doc_id
    score = scoreddoc.score

    scoreddoc_object = Scoredoc(doc_id, score)

    score_doc_dict[scoreddoc.query_id].append(scoreddoc_object)

score_doc_dict = dict(score_doc_dict)

print("Necessary dicts created!")

Necessary dicts created!


In [6]:
import pandas as pd
query_ids = [query.query_id for query in dataset.queries_iter()]
df = pd.DataFrame(query_ids, columns=["Query_ID"])
df

,Query_ID
0,123839
1,188629
2,13898
3,316959
4,515031
...,...
1439,896124
1440,12319
1441,4421
1442,296526


In [7]:
from helper import create_AP, create_ndcg, create_statistical_columns, print_columns
df = create_statistical_columns(df, qrels_dict, doc_dict, scores_of_all_queries)
df = create_AP(df, qrels_dict, doc_dict, scores_of_all_queries)
df = create_ndcg(df, doc_dict, scores_of_all_queries, score_doc_dict)

print_columns(df)

recall_5_mean: 14.639686703855874
recall_5_std: 14.53223970897621
recall_5_max: 83.33333333333334
recall_5_min: 0.0
recall_10_mean: 20.76798090604708
recall_10_std: 19.61319309097892
recall_10_max: 100.0
recall_10_min: 0.0
precision_5_mean: 29.930747922437675
precision_5_std: 22.919199945040646
precision_5_max: 100.0
precision_5_min: 0.0
precision_10_mean: 22.389196675900276
precision_10_std: 17.2715716550235
precision_10_max: 100.0
precision_10_min: 0.0
f_score_5_mean: 18.284844563791886
f_score_5_std: 16.779041729235384
f_score_5_max: 90.9090909090909
f_score_5_min: 0.0
f_score_10_mean: 19.463056209816017
f_score_10_std: 16.568674092887175
f_score_10_max: 88.88888888888889
f_score_10_min: 0.0
MAP_5: 0.11497240356894126
MAP_10: 0.14145452317090793
NDCG_5_mean: 0.4949990479623674
NDCG_5_std: 0.3536385014349461
NDCG_5_max: 1.0000000000000002
NDCG_5_min: 0.0
NDCG_10_mean: 0.4958507345878455
NDCG_10_std: 0.3511900772406139
NDCG_10_max: 1.0000000000000002
NDCG_10_min: 0.0
